In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import matplotlib.pyplot as plt
import pandas as pd

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import matplotlib.pyplot as plt
import pandas as pd

# Cargar tablas del modelo
dim_restaurant = spark.table("workspace.default.dim_restaurant")
fact_reviews = spark.table("workspace.default.fact_reviews")

print("Restaurantes:", dim_restaurant.count())
print("Reviews:", fact_reviews.count())

**Pergunta 1
Quais são os restaurantes mais populares e mais bem avaliados?**

Aqui, não queremos simplesmente buscar os restaurantes com a maior pontuação.

Queremos relacionar:

popularidade → quantidade de avaliações
qualidade percebida → pontuação média

In [0]:
restaurant_performance = (
    fact_reviews
    .join(
        dim_restaurant.select(
            "restaurant_id",
            "restaurant_name",
            "district"
        ),
        on="restaurant_id",
        how="inner"
    )
    .groupBy(
        "restaurant_id",
        "restaurant_name",
        "district"
    )
    .agg(
        F.count("review_id").alias("num_reviews"),
        F.round(F.avg("score"), 2).alias("average_score")
    )
)

display(
    restaurant_performance
    .orderBy(
        F.desc("num_reviews")
    )
    .limit(20)
)

In [0]:
restaurant_pd = restaurant_performance.toPandas()

plt.figure(figsize=(10, 6))

plt.scatter(
    restaurant_pd["num_reviews"],
    restaurant_pd["average_score"],
    alpha=0.5
)

plt.xlabel("Número de opiniones")
plt.ylabel("Valoración promedio")
plt.title("Popularidad vs. valoración de los restaurantes")

plt.ylim(0, 5)

plt.tight_layout()
plt.show()

**Pergunta 2: Quais distritos concentram a maior popularidade e qualidade?**

In [0]:
district_performance = (
    fact_reviews
    .join(
        dim_restaurant.select(
            "restaurant_id",
            "district"
        ),
        on="restaurant_id",
        how="inner"
    )
    .filter(
        F.col("district").isNotNull() &
        (F.trim(F.col("district")) != "")
    )
    .groupBy("district")
    .agg(
        F.countDistinct("restaurant_id").alias("num_restaurants"),
        F.count("review_id").alias("num_reviews"),
        F.round(F.avg("score"), 2).alias("average_score")
    )
    .filter(F.col("num_reviews") >= 100)
    .orderBy(F.desc("num_reviews"))
)

display(district_performance)

In [0]:
district_pd = district_performance.toPandas()

district_top = district_pd.sort_values(
    "num_restaurants",
    ascending=False
).head(15)

plt.figure(figsize=(11, 6))

plt.barh(
    district_top["district"],
    district_top["num_restaurants"]
)

plt.xlabel("Número de restaurantes")
plt.ylabel("Distrito")
plt.title("Distritos con mayor concentración de restaurantes")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

**Pergunta 3
Quais restaurantes se destacam em relação à média do seu distrito?**

In [0]:
district_avg = (
    fact_reviews
    .join(
        dim_restaurant.select(
            "restaurant_id",
            "district"
        ),
        on="restaurant_id",
        how="inner"
    )
    .groupBy("district")
    .agg(
        F.avg("score").alias("district_average_score")
    )
)

restaurant_vs_district = (
    restaurant_performance
    .join(
        district_avg,
        on="district",
        how="left"
    )
    .withColumn(
        "difference_from_district",
        F.round(
            F.col("average_score") -
            F.col("district_average_score"),
            2
        )
    )
    .filter(F.col("num_reviews") >= 100)
    .orderBy(F.desc("difference_from_district"))
)

display(
    restaurant_vs_district.limit(20)
)

In [0]:
comparison_pd = restaurant_vs_district.limit(15).toPandas()

plt.figure(figsize=(12, 7))

plt.barh(
    comparison_pd["restaurant_name"],
    comparison_pd["difference_from_district"]
)

plt.axvline(
    0,
    linewidth=1
)

plt.xlabel("Diferencia respecto al promedio del distrito")
plt.ylabel("Restaurante")
plt.title(
    "Restaurantes con mejor valoración respecto a su distrito"
)

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

Um valor de:

+0,50

significa que o restaurante está 0,50 pontos acima da média do seu distrito.

Isso permite encontrar restaurantes que realmente se destacam em relação aos seus concorrentes locais.

**Pergunta 4
Quais emoções caracterizam as opiniões positivas e negativas?**

Esta pergunta utiliza diretamente o modelo de emoções.

In [0]:
emotion_sentiment = (
    fact_reviews
    .filter(
        F.col("dominant_emotion").isNotNull() &
        (F.col("dominant_emotion") != "Not available") &
        F.col("sentiment").isNotNull() &
        (F.col("sentiment") != "Not available")
    )
    .groupBy(
        "sentiment",
        "dominant_emotion"
    )
    .agg(
        F.count("*").alias("num_reviews")
    )
)

display(
    emotion_sentiment.orderBy(
        "sentiment",
        F.desc("num_reviews")
    )
)

In [0]:
emotion_pd = emotion_sentiment.toPandas()

pivot_emotion = emotion_pd.pivot(
    index="dominant_emotion",
    columns="sentiment",
    values="num_reviews"
).fillna(0)

pivot_emotion.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.xlabel("Emoción predominante")
plt.ylabel("Número de opiniones")
plt.title(
    "Emociones predominantes según el sentimiento"
)

plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

**Pergunta 5 
Quais restaurantes combinam alta popularidade com alta satisfação?**

Esta será a nossa pergunta de destaque.

Calcularemos a porcentagem de avaliações positivas por restaurante.

In [0]:
restaurant_satisfaction = (
    fact_reviews
    .groupBy("restaurant_id")
    .agg(
        F.count("review_id").alias("num_reviews"),
        F.round(F.avg("score"), 2).alias("average_score"),
        F.sum(
            F.when(
                F.col("sentiment") == "Positive",
                1
            ).otherwise(0)
        ).alias("positive_reviews")
    )
    .withColumn(
        "positive_percentage",
        F.round(
            F.col("positive_reviews") /
            F.col("num_reviews") * 100,
            2
        )
    )
    .filter(F.col("num_reviews") >= 100)
)

restaurant_satisfaction = (
    restaurant_satisfaction
    .join(
        dim_restaurant.select(
            "restaurant_id",
            "restaurant_name",
            "district"
        ),
        on="restaurant_id",
        how="inner"
    )
)

display(
    restaurant_satisfaction.orderBy(
        F.desc("positive_percentage")
    ).limit(20)
)

In [0]:
satisfaction_pd = restaurant_satisfaction.toPandas()

plt.figure(figsize=(11, 7))

plt.scatter(
    satisfaction_pd["num_reviews"],
    satisfaction_pd["positive_percentage"],
    alpha=0.5
)

plt.xlabel("Número de opiniones")
plt.ylabel("Opiniones positivas (%)")
plt.title(
    "Popularidad vs. satisfacción de los clientes"
)

plt.ylim(0, 100)

plt.tight_layout()
plt.show()

O que buscamos?

A zona mais interessante é:

superior direita

porque representa:

Alta popularidade + alta satisfação

Estes restaurantes poderiam ser considerados as referências do mercado.